**Question** Are the labels balanced and is there evidence of leakage or shortcut learning?
**Method** Recompute balance, chi-square, KS, duplicate, single-feature, node-index, time-index, and label-shuffle checks from the saved tensors.
**Takeaway** Validity is an empirical question: this notebook reports the tests and ends with the simulator-to-production limitation.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from scipy.stats import chisquare, ks_2samp
ROOT = Path.cwd(); sys.path.extend([str(ROOT), str(ROOT/'notebooks'), str(ROOT/'python-ml')])
from notebooks.common.data import load_dataset, split_arrays
data = split_arrays(load_dataset()); train, val, test = data['train'], data['val'], data['test']
print(pd.DataFrame({'split':['train','val','test'], 'episodes':[len(v['failure']) for v in data.values()], 'failure_rate':[v['failure'].mean() for v in data.values()]}))
counts = np.bincount(train['root'][train['root'] < train['x'].shape[2]], minlength=train['x'].shape[2])
print('root counts:', counts, 'chi2:', chisquare(counts).statistic, 'p:', chisquare(counts).pvalue)
print('duplicate flat episodes:', len(train['flat']) - len(np.unique(train['flat'], axis=0)))

### What this tells us
These counts and tests are recomputed from the train/validation/test index files. A high chi-square p-value does not prove perfect uniformity; it only says this sample does not provide strong evidence against the tested uniform-null model.

In [ ]:
feature_names = ['cpu','memory','latency','error','in_degree','out_degree','criticality','queue']
rows=[]
for j, name in enumerate(feature_names):
    tr = train['x'][:,:,:,j].mean(axis=(1,2)); va = val['x'][:,:,:,j].mean(axis=(1,2)); te = test['x'][:,:,:,j].mean(axis=(1,2))
    rows.append({'feature':name, 'train_val_ks_p':ks_2samp(tr,va).pvalue, 'train_test_ks_p':ks_2samp(tr,te).pvalue})
ks_table = pd.DataFrame(rows); display(ks_table)

### What this tells us
The KS table measures distributional similarity feature by feature without using test labels for tuning. It can reveal covariate shift, but it cannot establish that synthetic telemetry matches a real cluster.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.dummy import DummyClassifier
from notebooks.common.metrics import failure_metrics
checks=[]
for name, column in [('node_index', np.tile(np.arange(train['x'].shape[2]), len(train['x'])).reshape(len(train['x']), -1).mean(axis=1)), ('time_index', np.full(len(train['x']), 4.5))]:
    model = DummyClassifier(strategy='prior') if np.unique(column).size == 1 else LogisticRegression(max_iter=200)
    Xtr = column.reshape(-1,1); model.fit(Xtr, train['failure']); pred = model.predict_proba(test['x'].mean(axis=(1,2))[:, :1] if name == 'time_index' else np.tile(np.arange(test['x'].shape[2]), len(test['x'])).reshape(len(test['x']),-1).mean(axis=1).reshape(-1,1))[:,1]
    checks.append({'shortcut':name, 'f1':f1_score(test['failure'], pred >= .5, zero_division=0)})
for j, name in enumerate(feature_names):
    model=LogisticRegression(max_iter=200).fit(train['x'][:,:,:,j].mean(axis=(1,2)).reshape(-1,1), train['failure'])
    p=model.predict_proba(test['x'][:,:,:,j].mean(axis=(1,2)).reshape(-1,1))[:,1]
    checks.append({'shortcut':name, 'f1':failure_metrics(test['failure'],p)['f1']})
print(pd.DataFrame(checks))

### What this tells us
Each shortcut score is evaluated independently on the held-out split. A single feature can be predictive without being a complete explanation; the relevant comparison is against the full-feature model in the separate model-comparison workflow.

In [ ]:
from sklearn.linear_model import LogisticRegression
from notebooks.common.metrics import failure_metrics
rng=np.random.default_rng(42); shuffled=train['failure'].copy(); rng.shuffle(shuffled)
model=LogisticRegression(max_iter=300).fit(train['flat'], shuffled)
print('label-shuffled test metrics:', failure_metrics(test['failure'], model.predict_proba(test['flat'])[:,1]))
print('RCAEval EDA is available through dataset/preprocess_rcaeval.py; this synthetic simulator audit is separate.')

### What this tells us
Label shuffling removes the training relationship while preserving the feature distribution, so performance should fall toward the class-prior level. Synthetic simulator checks are separate from RCAEval and do not establish production detector performance.

## Limitation
The dataset is a controlled simulator. These checks can expose internal imbalance, duplication, distribution shift, and obvious leakage, but they cannot close the simulator-versus-real-cluster gap. Real telemetry, changing workloads, instrumentation faults, and operational interventions remain external validity risks.